# 02 · Campaign — agonist mini-proteins engaging the chosen receptor surfaces

**Standard slot:** *design campaign.* **For Project 13 this is the core:** design de novo agonist
mini-proteins steered onto the **IL-2Rβ + γc signaling surface** (they must bridge both chains to
dimerize the receptor), then score **every** design against **each of the three subunits separately**
to build the selectivity data (D2):
- **RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences (primary).
- (optional foil) **BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.

Then score with **AF2-Multimer** *per subunit* (`pae_interaction` to α, β, γc — the selectivity profile).

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster), and
> modeling against **three** subunits roughly **triples** the AF2-Multimer cost. Free **T4** = a *small
> fallback* (small RFdiffusion batch + ESMFold triage on β/γc; FreeBindCraft, small `num_designs`). The
> cells below run on the deterministic **mock** backend so the plumbing executes anywhere; the real
> calls + A100 notes are shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change!)

The design tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log it).
The generation itself needs an A100; this check needs nothing.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # binder mode; pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   BindCraft      https://github.com/martinpacesa/BindCraft         # one-shot hallucination; pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft      # free-tier fallback — VERIFY it exists; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer (per subunit); pin <commit>
PINNED = {
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")

## 1 · Define the campaign

Same target + per-subunit hotspots as notebook 01. Set honest campaign sizes; the cells run on `mock`
so they execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and **shrink the
numbers on a T4** (small RFdiffusion batch, FreeBindCraft). Remember AF2-Multimer runs **per subunit**.

In [ ]:
import cytokine_tools as ct
import pandas as pd

TARGET = "IL2R_beta_gamma"
HOTSPOTS = ct.parse_hotspots("B41,B42,C100,C102")   # EXAMPLE — replace with your verified β/γc residues
ENGAGE = ct.ENGAGE_DEFAULT        # ("IL2Rb","gammaC") -> the signaling pair to engage
SPARE  = ct.SPARE_DEFAULT         # ("IL2Ra",)         -> CD25, the chain to spare
SUBUNITS = ct.SUBUNITS            # ("IL2Ra","IL2Rb","gammaC") -> AF2-Multimer is run vs EACH

# Honest campaign sizes (catalog): RFdiffusion 500-1000 backbones, BindCraft 50-200.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4

TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer, run per subunit) on Colab

print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"AF2-Multimer: tool={TOOL_AF2}  (run vs EACH subunit: {SUBUNITS})")
print("hotspots    :", HOTSPOTS, " | ENGAGE", ENGAGE, " SPARE", SPARE)

## 2 · Primary paradigm — RFdiffusion binder campaign → ProteinMPNN

Diffuse mini-protein backbones docked at the β/γc hotspots (supply **both** receptor chains so the
design can bridge them → dimerize → signal), then ProteinMPNN designs sequences, then AF2-Multimer
re-predicts each complex **against each subunit**. On A100 this is 500–1000 backbones (per-backbone hit
rate is low — that is normal). The `mock` backend stands in for the whole chain.

In [ ]:
# Real call (Colab, A100): ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). Supply BOTH β and γc chains so the
#   mini-protein can bridge them. AF2-Multimer (×3 subunits) is the slow step. See MANUAL.md §2 / scripts.
rfdiff = ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
ct.score_designs(rfdiff, tool=TOOL_AF2, engage=ENGAGE, spare=SPARE)   # fills per-subunit pae + metrics
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
ex = rfdiff[0]
print("example:", ex.design_id, "per-subunit pae =", ex.pae_by_subunit)

## 3 · Optional foil — BindCraft campaign

BindCraft hallucinates a mini-protein with AF2-Multimer in the loop. Useful as a second paradigm/foil
at the same surface; we still re-score it against each subunit so the selectivity analysis is
apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [ ]:
# Real call (Colab, A100): ct.generate_agonists_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/cytokine_tools.py TODOs.
bindcraft = ct.generate_agonists_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
ct.score_designs(bindcraft, tool=TOOL_AF2, engage=ENGAGE, spare=SPARE)
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
ex = bindcraft[0]
print("example:", ex.design_id, "per-subunit pae =", ex.pae_by_subunit)

## 4 · Assemble + persist the pool (with per-subunit metrics)

Write one tidy CSV with the **per-subunit `pae_to_*`** columns (the selectivity data) plus the standard
binder metrics. These feed notebook 03 (the shared filter) and notebook 04 (the selectivity profile).
We add an EXAMPLE physics column (`rosetta_dG`) so the binder physics layer has something to act on in
the dry run — on Colab these come from FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC.

In [ ]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (ct._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        pae = d.pae_by_subunit or {}
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, scrmsd=d.scrmsd, shape_complementarity=d.shape_complementarity,
            pae_to_alpha=pae.get("IL2Ra"), pae_to_beta=pae.get("IL2Rb"), pae_to_gamma=pae.get("gammaC"),
            pae_interaction=d.pae_interaction,        # worst engaged-subunit pae (β/γc)
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            hotspot_overlap=ct.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_rf = pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
df_bc = pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
combined = pd.concat([df_rf, df_bc], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined[["design_id","paradigm","pae_to_alpha","pae_to_beta","pae_to_gamma","pae_interaction"]].head(4)

## D2 checklist
- [ ] RFdiffusion-binder pool generated at honest scale (500–1000 backbones → ProteinMPNN on A100).
- [ ] (optional foil) BindCraft pool generated (50–200; FreeBindCraft/small on T4).
- [ ] Every design scored by AF2-Multimer **against each of α/β/γc** (`pae_to_*` parsed); pool written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the engaged subunits.